# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

In [3]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
deepseek_url = "https://api.deepseek.com"


gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)

In [ ]:
# open source models
# qwen2.5-coder (Alibaba)
# deepseek-coder-v2 (DeepSeek)
# gpt-oss:20b (OpenAI)
# qwen/qwen3-coder-30b-a3b-instruct (Alibaba)
# openai/gpt-oss-120b (OpenAI)
# model:clientlibrary
models = ["qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b" ]
clients = {"openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": ollama}



In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.19045',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': 'Intel(R) Core(TM) i3-4030U CPU @ 1.90GHz',
  'cores_logical': 4,
  'cores_physical': 2,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [7]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': False,
 'rustc': {'path': '',
  'version': '',
  'host_triple': '',
  'release': '',
  'commit_hash': ''},
 'cargo': {'path': '', 'version': ''},
 'rustup': {'path': '',
  'version': '',
  'active_toolchain': '',
  'default_toolchain': '',
  'toolchains': [],
  'targets_installed': []},
 'rust_analyzer': {'path': ''},
 'env': {'CARGO_HOME': '',
  'RUSTUP_HOME': '',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': []}

In [10]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = groq.chat.completions.create(model=models[4], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

## 1️⃣  Do you need a Rust tool‑chain?  
Yes – the system report shows **no Rust tool‑chain installed** (`rustc`, `cargo` and `rustup` are missing).  

You’ll have to install the official Rust tool‑chain first.  

---

## 2️⃣  Install the Rust tool‑chain on Windows (the simplest way)

| Method | One‑liner command | What it does |
|--------|------------------|--------------|
| **Winget** (built‑in Windows package manager) | `winget install --id Rustlang.Rustup -e --source winget` | Installs `rustup‑init.exe` which then bootstraps the stable Rust tool‑chain. |
| **Direct rustup installer** (if you prefer the classic installer) | Download <https://win.rustup.rs> and run the `.exe` (just click “Next → Install”). | Same result – `rustup`, `cargo` and `rustc` end up in `%USERPROFILE%\.cargo\bin`. |

### After the installer finishes

```powershell
# Add Cargo’s bin directory to your PATH for the current session
$env:Path += ";$HOME\.cargo\bin"

# Verify the installation
rustc --version   # e.g. rustc 1.77.0 (2024‑05‑15)
cargo --version  # e.g. cargo 1.77.0 (2024‑05‑15)
rustup --version  # e.g. rustup 1.27.0 (2024‑05‑15)
```

If you used **winget**, the installer already puts the Cargo bin folder on your system `PATH`, so you can open a new Command Prompt / PowerShell window and run the two verification commands directly.

---

## 3️⃣  Compile **main.rs** for **maximum runtime performance**

The fastest executable (at the cost of longer compile time) is obtained by:

* Full optimisations (`-C opt-level=3`)
* Targeting the *native* CPU (`-C target-cpu=native`)
* Enabling **Link‑Time Optimisation** (`-C lto=yes`)
* Using a **single code‑gen unit** (`-C codegen-units=1`) – helps LTO
* **Abort** on panic (`-C panic=abort`) – removes panic‑handling overhead
* No debug info (`-C debuginfo=0`)

All of those flags can be passed to `rustc` directly (no Cargo needed for a single file).

### Full command line

```text
rustc main.rs \
    -C opt-level=3 \
    -C target-cpu=native \
    -C lto=yes \
    -C codegen-units=1 \
    -C panic=abort \
    -C debuginfo=0 \
    -C strip=debuginfo \
    -o main_opt.exe
```

*`-C strip=debuginfo`* removes any remaining symbols, making the binary a little smaller (optional).

---

## 4️⃣  Using Python’s `subprocess` to compile **and** run

Below is a minimal, self‑contained snippet that:

1. **Compiles** `main.rs` with the performance‑tuned flags shown above.  
2. **Runs** the resulting `main_opt.exe`.  
3. Returns the program’s `stdout`.

```python
import subprocess
import shlex
from pathlib import Path

# ----------------------------------------------------------------------
# 1️⃣  Compile the Rust source
# ----------------------------------------------------------------------
compile_cmd = [
    "rustc", "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=yes",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "-C", "debuginfo=0",
    "-C", "strip=debuginfo",
    "-o", "main_opt.exe"
]

compile_result = subprocess.run(
    compile_cmd,
    check=True,
    text=True,
    capture_output=True  # captures rustc's stdout/stderr (useful for debugging)
)

# ----------------------------------------------------------------------
# 2️⃣  Execute the compiled binary
# ----------------------------------------------------------------------
run_cmd = ["./main_opt.exe"]   # Windows can also use "main_opt.exe"

run_result = subprocess.run(
    run_cmd,
    check=True,
    text=True,
    capture_output=True
)

# ----------------------------------------------------------------------
# 3️⃣  Return the program output
# ----------------------------------------------------------------------
print(run_result.stdout)   # or `return run_result.stdout` inside a function
```

### Why these exact lists?

* `subprocess.run` expects **a list of arguments** (no shell needed).  
* Using the explicit `-C` flags gives you the *maximum* runtime speed on this particular machine.  
* The output binary name (`main_opt.exe`) is passed to the **run** step, so there’s no need for a temporary Cargo project.

---

## 5️⃣  Quick sanity‑check

After installing Rust, you can test everything manually in a Command Prompt / PowerShell window:

```powershell
# Compile
rustc main.rs -C opt-level=3 -C target-cpu=native -C lto=yes -C codegen-units=1 -C panic=abort -C debuginfo=0 -C strip=debuginfo -o main_opt.exe

# Run
.\main_opt.exe
```

If the program prints the expected output, the Python snippet will work identically.

---

### TL;DR (the cheat‑sheet)

1. **Install Rust** (one‑liner):  

   ```powershell
   winget install --id Rustlang.Rustup -e --source winget
   ```

2. **Compile** (fastest runtime):  

   ```powershell
   rustc main.rs -C opt-level=3 -C target-cpu=native -C lto=yes -C codegen-units=1 -C panic=abort -C debuginfo=0 -C strip=debuginfo -o main_opt.exe
   ```

3. **Run**:  

   ```powershell
   .\main_opt.exe
   ```

4. **Python wrapper** – see the code block above (`compile_cmd` & `run_cmd`).  

That’s all you need to get a highly‑optimised Rust executable compiled and executed from Python on your Windows 10 (AMD64) machine. Happy coding! 🚀

In [12]:
compile_cmd = [
    "rustc", "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=yes",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "-C", "debuginfo=0",
    "-C", "strip=debuginfo",
    "-o", "main_opt.exe"
]

run_command = [".\main_opt.exe"]

<>:13: SyntaxWarning: invalid escape sequence '\m'
<>:13: SyntaxWarning: invalid escape sequence '\m'
C:\Users\HP\AppData\Local\Temp\ipykernel_5272\1868414929.py:13: SyntaxWarning: invalid escape sequence '\m'
  run_command = [".\main_opt.exe"]


## main task

In [25]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

# def user_prompt_for(python):
#     return f"""
# Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
# The system information is:
# {system_info}
# Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
# {compile_cmd}
# Respond only with C++ code.
# Python code to port:

# ```python
# {python}
# ```
# """

def user_prompt_for(python, sys_info, comp_cmd):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The target system information is:
{sys_info}

Your response will be written to a file called main.cpp and then compiled and executed; the compilation configuration intended is:
{comp_cmd}

Respond strictly with valid C++ code wrapped in a markdown code block.
Python code to port:

```python
{python}
"""

In [26]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python,system_info,compile_cmd)}
    ]

In [16]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [36]:
# def port(model, python):
#     client = clients[model]
#     reasoning_effort = "high" if 'gpt' in model else None
#     response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
#     reply = response.choices[0].message.content
#     reply = reply.replace('```cpp','').replace('```','')
#     write_output(reply)
#     return reply

def port(model, python):
    client = clients[model]
    
    # Safely assign reasoning params only if explicitly required by specific endpoints
    kwargs = {
        "model": model, 
        "messages": messages_for(python),
        "temperature": 0.0, # 0.0 forces strict deterministic code translation
    }
    
    # Explicitly check for reasoning-effort support strings
    if 'gpt' in model and 'groq' not in str(client.base_url):
        kwargs["reasoning_effort"] = "high"
        
    print(f"Sending request to {model}...")
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content
    
    # Diagnostic print if it returns empty
    if not reply:
        print(f"❌ Error: {model} returned an empty payload string block.")
        return ""
        
    print(f"Response received from {model} ({len(reply)} chars). Parsing...")

    # Robust Multi-stage Code Block Extraction Logic
    if "```cpp" in reply:
        reply = reply.split("```cpp")[1].split("```")[0].strip()
    elif "```" in reply:
        reply = reply.split("```")[1].split("```")[0].strip()
    else:
        # If the model skipped code blocks and only returned plain text, 
        # clean out any accidental markdown artifacts safely
        reply = reply.replace('```cpp', '').replace('```', '').strip()
        
    write_output(reply)
    print(f"✅ Code successfully written to main.cpp!")
    return reply
    

In [28]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [37]:
port("openai/gpt-oss-120b",pi)

Sending request to openai/gpt-oss-120b...
Response received from openai/gpt-oss-120b (927 chars). Parsing...
✅ Code successfully written to main.cpp!


'#include <bits/stdc++.h>\nusing namespace std;\n\ninline double calculate(uint64_t iterations, double p1, double p2) {\n    double result = 1.0;\n    for (uint64_t i = 1; i <= iterations; ++i) {\n        double i_d = static_cast<double>(i);\n        double j = i_d * p1 - p2;\n        result -= 1.0 / j;\n        j = i_d * p1 + p2;\n        result += 1.0 / j;\n    }\n    return result;\n}\n\nint main() {\n    const uint64_t ITER = 200\'000\'000ULL;\n    const double PARAM1 = 4.0;\n    const double PARAM2 = 1.0;\n\n    auto start = chrono::high_resolution_clock::now();\n    double result = calculate(ITER, PARAM1, PARAM2) * 4.0;\n    auto end = chrono::high_resolution_clock::now();\n\n    double elapsed = chrono::duration<double>(end - start).count();\n\n    cout.setf(ios::fixed);\n    cout << setprecision(12) << "Result: " << result << "\\n";\n    cout << setprecision(6) << "Execution Time: " << elapsed << " seconds\\n";\n    return 0;\n}'

In [38]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [39]:
run_python(pi)

'Result: 3.141592656089\nExecution Time: 164.463903 seconds\n'

In [46]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [47]:
ui.close()

Closing server running on port: 7861
